# Build a RAG dataset over 10-K filings

Use full-text search to find the right 10-Ks and passages, then fetch the documents from sec.gov to chunk and embed for retrieval.

Read it with charts at https://www.quantisnow.com/sec-api/guides/rag-dataset-10-k. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Retrieval-augmented generation needs two things: the right documents, and the right passages from them. Full-text search does the first part directly and hands back matching passages and the filing index link for each, which is where the full document lives.

## Find the filings and passages

### Which 10-Ks talk about artificial intelligence?

AI turns up in 10-Ks as a product, a cost, and more and more as a risk factor. The passages show which.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"artificial intelligence\" OR \"generative AI\"",
  "formTypes": ["10-K", "10-K/A"],
  "sort": "filedAt:desc",
  "size": 5,
  "facets": true
}'
```

Response (real, first 2 of 80):

```json
{
  "total": {
    "value": 80,
    "relation": "eq"
  },
  "took": 220,
  "data": [
    {
      "id": "0001681622-26-000086",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001681622-26-000086",
      "formType": "10-K",
      "filedAt": "2026-11-23T16:40:29-05:00",
      "cik": "0001681622",
      "companyName": "Varex Imaging Corporation",
      "ticker": "VREX",
      "tickers": [
        "VREX"
      ],
      "items": [],
      "description": "SEC Form 10-K filed by Varex Imaging Corporation",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1681622/000168162226000086/0001681622-26-000086-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-k-filed-by-varex-imaging-corporation-6804221",
      "score": null,
      "highlights": [
        "Failure to develop, validate, adopt, govern, or properly manage <em>artificial intelligence</em> (\"AI\") technology, its use and applications, including AI used in products, engineering, manufacturing, quality, regulatory, service"
      ]
    },
    {
      "id": "0001628280-26-075495",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001628280-26-075495",
      "formType": "10-K",
      "filedAt": "2026-11-23T16:13:24-05:00",
      "cik": "0000052988",
      "companyName": "Jacobs Solutions Inc.",
      "ticker": "J",
      "tickers": [
        "J"
      ],
      "items": [],
      "description": "SEC Form 10-K filed by Jacobs Solutions Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/52988/000162828026075495/0001628280-26-075495-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-k-filed-by-jacobs-solutions-inc-6804018",
      "score": null,
      "highlights": [
        "We harness data, digital platforms and <em>artificial intelligence</em> (AI)-enabled solutions to help clients operate more efficiently, safely and intelligently, while enhancing resilience and supporting more sustainable outcomes",
        "In addition, the rapid evolution and increased adoption of <em>artificial intelligence</em>, machine learning, data science and similar technologies (collectively, \"AI\") may intensify our cybersecurity risks (e.g., AI used to develop",
        "We use <em>artificial intelligence</em>, machine learning, data science and similar technologies in our business, and challenges with properly managing such technologies could result in reputational harm, competitive harm, and legal"
      ]
    }
  ],
  "facets": {
    "formType": [
      {
        "value": "10-K",
        "count": 79
      },
      {
        "value": "10-K/A",
        "count": 1
      }
    ],
    "month": [
      {
        "value": "2026-11",
        "count": 74
      },
      {
        "value": "2026-10",
        "count": 6
      }
    ],
    "companyName": [
      {
        "value": "AMCON Distributing Company",
        "count": 1
      },
      {
        "value": "Adient plc",
        "count": 1
      },
      {
        "value": "Air Products and Chemicals Inc.",
        "count": 1
      },
      {
        "value": "Alico Inc.",
        "count": 1
      },
      {
        "value": "Apple Inc.",
        "count": 1
      },
      {
        "value": "Arrowhead Pharmaceuticals Inc.",
        "count": 1
      },
      {
        "value": "Ashland Inc.",
        "count": 1
      },
      {
        "value": "AutoZone Inc.",
        "count": 1
      },
      {
        "value": "Azenta Inc.",
        "count": 1
      },
      {
        "value": "BellRing Brands Inc.",
        "count": 1
      }
    ],
    "source": [
      {
        "value": "SEC",
        "count": 80
      }
    ]
  }
}
```

## Collect them in Python

In [ ]:
import json, os, requests

API = "https://www.quantisnow.com/api/v1/full-text"
H = {"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"}

records, start = [], 0
while start < 500:
    page = requests.post(API, headers=H, json={
        "query": '"artificial intelligence" AND "risk"', "formTypes": ["10-K"],
        "sort": "filedAt:desc", "size": 50, "from": start,
    }).json()
    for f in page["data"]:
        for i, passage in enumerate(f["highlights"]):
            records.append({
                "id": f"{f['accessionNo']}-{i}",
                "text": passage.replace("<em>", "").replace("</em>", ""),
                "ticker": f["ticker"], "company": f["companyName"],
                "filedAt": f["filedAt"], "source": f["linkToFilingIndex"],
            })
    start += 50
    if start >= page["total"]["value"]:
        break

with open("passages.jsonl", "w") as out:
    for r in records:
        out.write(json.dumps(r) + "\n")
print(len(records), "passages")

Passages are up to three per filing, enough for a small index with sources. For whole documents, follow `linkToFilingIndex` to sec.gov and download the main document; the SEC asks for a User-Agent with your contact details and at most 10 requests a second. Keep the accession number on every chunk so each answer can cite its filing.

## Or skip the index

If the goal is an assistant that answers from filings, an [MCP connection](https://www.quantisnow.com/sec-api/mcp) gives it `full_text_search` directly, so it can search at question time instead of from a fixed index.

Related: [Search 10-K text](https://www.quantisnow.com/sec-api/guides/search-10-k-text), [Keyword search in Python](https://www.quantisnow.com/sec-api/guides/search-sec-filings-python).